In [ ]:
# ============================================================
# PACKAGE 12 : CITIZEN HEALTH ADVISORY AGENT
# VERSION 3.0
# PART 2/3
#
# Purpose:
# - Generate citizen health advisories
# - Create ward-level recommendations
# - Prepare frontend JSON
#
# Input:
#   Ward_Health_Risk.csv
#
# Output:
#   Citizen_Health_Advisory_Report.csv
#   Frontend_Health_Advisory.json
#
# ============================================================


import os
import json
import pandas as pd


print("="*75)
print("PACKAGE 12 VERSION 3.0")
print("CITIZEN HEALTH ADVISORY AGENT")
print("PART 2/3")
print("="*75)



# ============================================================
# CONFIGURATION
# ============================================================


OUTPUT_DIR = (
    r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Citizen_Health_Advisory\Part2"
)


WARD_HEALTH_FILE = (r"C:\Users\91863\Desktop\AIR QUALITY INTELLIGENCE\Citizen_Health_Advisory\Part2"


REPORT_FILE = os.path.join(
    OUTPUT_DIR,
    "Citizen_Health_Advisory_Report.csv"
)


FRONTEND_JSON = os.path.join(
    OUTPUT_DIR,
    "Frontend_Health_Advisory.json"
)



# ============================================================
# LOAD HEALTH DATA
# ============================================================


print("\nLoading Ward Health Risk Data...")


if not os.path.exists(WARD_HEALTH_FILE):

    raise FileNotFoundError(
        "Ward_Health_Risk.csv not found. Run Part 1 first."
    )



health_df = pd.read_csv(
    WARD_HEALTH_FILE
)



print(
    "Loaded Shape:",
    health_df.shape
)



# ============================================================
# ADVISORY GENERATION FUNCTIONS
# ============================================================



def generate_health_message(row):


    level = row["Health_Risk_Level"]

    source = row["Dominant_Source"]


    if level == "Critical":

        return (
            f"Critical air pollution risk detected. "
            f"Residents should avoid outdoor exposure. "
            f"Main contributor: {source}. "
            f"Authorities should initiate immediate mitigation."
        )


    elif level == "High":

        return (
            f"High pollution health risk. "
            f"Sensitive groups should limit outdoor activity. "
            f"Priority source: {source}. "
            f"Increase monitoring and enforcement."
        )


    elif level == "Moderate":

        return (
            f"Moderate pollution risk. "
            f"Children, elderly and respiratory patients "
            f"should take precautions. "
            f"Monitor {source} related activities."
        )


    else:

        return (
            "Air quality conditions are relatively acceptable. "
            "Continue regular monitoring."
        )




def generate_action(row):


    source = row["Dominant_Source"]


    if source == "Traffic Emission":

        return (
            "Traffic regulation, vehicle inspection "
            "and congestion control recommended."
        )


    elif source == "Construction Activity":

        return (
            "Inspect construction sites and enforce "
            "dust suppression measures."
        )


    elif source == "Industrial Pollution":

        return (
            "Inspect industrial emission sources "
            "and verify compliance."
        )


    else:

        return (
            "Continue pollution monitoring."
        )



# ============================================================
# GENERATE ADVISORY
# ============================================================



print("\nGenerating Citizen Advisories...")



health_df["Citizen_Message"] = (

    health_df.apply(
        generate_health_message,
        axis=1
    )

)



health_df["Recommended_Action"] = (

    health_df.apply(
        generate_action,
        axis=1
    )

)



# ============================================================
# PRIORITY SCORE
# ============================================================


risk_priority = {

    "Critical": 4,

    "High": 3,

    "Moderate": 2,

    "Low": 1

}



health_df["Priority_Level"] = (

    health_df["Health_Risk_Level"]

    .map(risk_priority)

    .fillna(0)

)



# ============================================================
# SORT HOTSPOTS
# ============================================================



health_df = health_df.sort_values(

    by=[
        "Priority_Level",
        "Health_Risk_Score"
    ],

    ascending=False

)



# ============================================================
# SAVE REPORT
# ============================================================


health_df.to_csv(

    REPORT_FILE,

    index=False

)



print(
    "\nSaved:"
)

print(
    REPORT_FILE
)



# ============================================================
# FRONTEND JSON CREATION
# ============================================================



frontend_records = []



for _, row in health_df.iterrows():


    frontend_records.append(

        {

            "ward_name":
                row.get(
                    "Ward_Name",
                    "Unknown"
                ),


            "ward_no":
                row.get(
                    "Ward_No",
                    "Unknown"
                ),


            "health_risk_score":
                round(
                    float(
                        row["Health_Risk_Score"]
                    ),
                    2
                ),


            "risk_level":
                row["Health_Risk_Level"],


            "dominant_source":
                row["Dominant_Source"],


            "confidence":
                row["Confidence"],


            "citizen_message":
                row["Citizen_Message"],


            "recommended_action":
                row["Recommended_Action"]

        }

    )



frontend_output = {


    "model":

        "Citizen Health Advisory Agent",


    "total_wards":

        len(frontend_records),


    "highest_priority_wards":

        frontend_records[:10],


    "all_ward_advisories":

        frontend_records


}



with open(

    FRONTEND_JSON,

    "w",

    encoding="utf-8"

) as f:


    json.dump(

        frontend_output,

        f,

        indent=4

    )



print(
    "\nSaved:"
)

print(
    FRONTEND_JSON
)



print("\n")
print("="*60)
print("PART 2 COMPLETED SUCCESSFULLY")
print("="*60)



# Available for Part 3:
#
# health_df
#
# Outputs:
# Citizen_Health_Advisory_Report.csv
# Frontend_Health_Advisory.json
